# Aprender a decidir: MDP, Bellman y un mundo de casillas

**Facsímil 10 · Aprendizaje por refuerzo** — capítulo 1 (MDP, políticas, retorno y Bellman).

Decidir bajo incertidumbre, con consecuencias que se **encadenan**, es lo que hacen un robot aspirador, un
GPS que recalcula la ruta, un ascensor que decide a qué planta ir o un agente que planea pasos. El marco
matemático de ese problema es el **proceso de decisión de Markov** (MDP), y su ecuación fundamental es la
de **Bellman**. En este cuaderno resuelves un pequeño mundo de casillas donde un robot busca la salida
evitando una trampa, **sin enseñarle el camino**: solo le das las recompensas y dejas que Bellman calcule,
casilla a casilla, lo que vale estar en cada sitio. De ahí sale sola la **política óptima**. Es el
esqueleto de todo el aprendizaje por refuerzo.

### Qué vas a aprender
- Qué es un **MDP**: estados, acciones, recompensas, transiciones (con algo de azar) y descuento.
- La **ecuación de Bellman** y cómo iterarla (*value iteration*) hace emerger la política sin programar
  el camino.
- A **ver** los valores y la política dibujados, y a entender cómo el valor negativo de la trampa se
  **propaga** a sus vecinos.
- Cómo el **descuento** γ, el **coste del paso** y el **azar** del entorno cambian lo que es prudente hacer,
  con un experimento medido para cada uno.
- A **simular trayectorias** con la política y a comprobar que *value iteration* y *policy iteration*
  llegan al mismo sitio por caminos distintos.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# Mundo 3x4. +1 = salida buena, -1 = trampa, # = muro.
FILAS, COLS = 3, 4
META, TRAMPA, MURO = (0,3), (1,3), (1,1)
RECOMPENSA_PASO = -0.04          # moverse cuesta un poquito (incita a no dar rodeos)
GAMMA = 0.9                       # cuanto importa el futuro (descuento)
terminales = {META: 1.0, TRAMPA: -1.0}
ACCIONES = {"^":(-1,0), "v":(1,0), "<":(0,-1), ">":(0,1)}

def estados(): return [(f,c) for f in range(FILAS) for c in range(COLS) if (f,c) != MURO]
def mover(s, a):
    f, c = s[0]+ACCIONES[a][0], s[1]+ACCIONES[a][1]
    return (f,c) if (0<=f<FILAS and 0<=c<COLS and (f,c)!=MURO) else s   # chocar = te quedas

print("Mundo 3x4 con salida (+1), trampa (-1) y un muro (#). El robot NO sabe el camino.")
print("Mapa (S=salida, T=trampa, #=muro, .=casilla normal, R=inicio del robot):")
for f in range(FILAS):
    fila = []
    for c in range(COLS):
        s = (f,c)
        if s == META: fila.append("S")
        elif s == TRAMPA: fila.append("T")
        elif s == MURO: fila.append("#")
        elif s == (2,0): fila.append("R")
        else: fila.append(".")
    print("   " + " ".join(fila))


## 1. Qué es un MDP

Un proceso de decisión de Markov tiene cinco ingredientes. Los vemos con el robot aspirador como ejemplo:

- **Estados:** las situaciones posibles (aquí, las casillas; en el aspirador, en qué habitación está).
- **Acciones:** lo que el agente puede hacer en cada estado (moverse arriba/abajo/izquierda/derecha).
- **Transiciones:** a qué estado lleva cada acción, posiblemente **con azar** (el suelo resbala, una rueda
  patina: no siempre acabas donde querías).
- **Recompensas:** el premio o castigo de cada paso (+1 salir, -1 caer, -0,04 por moverse, igual que la
  batería que se gasta con cada movimiento).
- **Descuento γ:** cuánto valen las recompensas futuras frente a las inmediatas. γ cercano a 1 = el
  agente piensa a largo plazo.

La propiedad de **Markov**: el futuro depende solo del estado actual, no de cómo llegaste a él. Resolver
el MDP es encontrar la **política** (qué hacer en cada estado) que maximiza la recompensa total
esperada a largo plazo.


In [ ]:
def resultados(s, a):
    # 80% la accion elegida, 10% cada perpendicular (mundo resbaladizo)
    perp = {"^":["<",">"], "v":["<",">"], "<":["^","v"], ">":["^","v"]}[a]
    return [(0.8, mover(s,a)), (0.1, mover(s,perp[0])), (0.1, mover(s,perp[1]))]

print("Si el robot intenta ir a la derecha, el 80% de las veces lo logra, pero el 20% resbala de lado.")
print("Ejemplo desde (2,0) intentando '>':", resultados((2,0), ">"))
print("Ejemplo desde (2,2) intentando '^':", resultados((2,2), "^"))
print("\nFijate: la suma de probabilidades de cada accion es 1. El azar no quita masa, la reparte.")


## 2. La ecuación de Bellman, iterada

El **valor** de una casilla es la mejor recompensa que puedes esperar desde ahí: lo que ganas al moverte
más el valor (descontado por γ) de donde acabas, promediado sobre el azar. En fórmula:
$$ V(s) = \max_a \sum_{s'} P(s'\mid s,a)\,\big[\,r + \gamma\,V(s')\,\big] $$
No sabemos $V$ de antemano, así que la calculamos **iterando**: empezamos con ceros y aplicamos la
fórmula una y otra vez hasta que los valores dejan de cambiar (*value iteration*). Es un punto fijo que
converge. Guardamos cuánto cambian los valores en cada vuelta para *ver* esa convergencia.


In [ ]:
def valor_de(s, a, V, gamma, coste):
    return sum(p*(coste + gamma*V[s2]) for p, s2 in resultados(s, a))

V = {s: 0.0 for s in estados()}
for s, r in terminales.items(): V[s] = r
historia = []

for iteracion in range(1, 200):
    delta = 0
    for s in estados():
        if s in terminales: continue
        mejor = max(valor_de(s, a, V, GAMMA, RECOMPENSA_PASO) for a in ACCIONES)
        delta = max(delta, abs(mejor - V[s])); V[s] = mejor
    historia.append(delta)
    if delta < 1e-6: break

print(f"Convergio en {iteracion} iteraciones.\n")
print("VALOR de cada casilla (lo que vale estar ahi):")
for f in range(FILAS):
    print("  " + "  ".join(f"{V.get((f,c),0):+.2f}" if (f,c)!=MURO else "  ##  " for c in range(COLS)))


## 3. La convergencia, dibujada

Cada vuelta de Bellman cambia los valores un poco menos que la anterior. Si dibujamos ese «cuánto cambió»
(el *delta*) en escala logarítmica, vemos una recta que baja: la iteración se acerca al punto fijo a
ritmo **geométrico**. Cuando el cambio es despreciable, paramos: ya no hay nada que afinar.


In [ ]:
plt.figure(figsize=(6.2, 3.2))
plt.semilogy(range(1, len(historia)+1), historia, "-o", color="black", markersize=4)
plt.xlabel("iteracion de Bellman"); plt.ylabel("cambio maximo (escala log)")
plt.title("Value iteration converge a ritmo geometrico"); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"El cambio cae de {historia[0]:.3f} en la primera vuelta a {historia[-1]:.1e} en la ultima.")


## 4. La política óptima sale sola

Una vez sabes cuánto vale cada casilla, la decisión en cada una es trivial: ir hacia la vecina más
valiosa (en esperanza). Nadie programó el camino; **emergió** de las recompensas y de Bellman. Es la
diferencia entre *decir* el camino y *deducirlo* del valor.


In [ ]:
def politica_de(V, gamma, coste):
    return {s: max(ACCIONES, key=lambda a: valor_de(s, a, V, gamma, coste))
            for s in estados() if s not in terminales}

POL = politica_de(V, GAMMA, RECOMPENSA_PASO)

def dibuja_politica(pol):
    for f in range(FILAS):
        fila = []
        for c in range(COLS):
            s = (f,c)
            if s == MURO: fila.append(" # ")
            elif s == META: fila.append(" +1")
            elif s == TRAMPA: fila.append(" -1")
            else: fila.append(f"  {pol[s]}")
        print("  " + " ".join(fila))

print("POLITICA optima (que hacer en cada casilla):")
dibuja_politica(POL)
print("\nLas flechas evitan la trampa y rodean el muro. Nadie las programo: salen del valor.")


**Lo que acaba de pasar.** No le dijimos al robot «ve a la derecha y luego arriba». Le dimos
recompensas y la ecuación de Bellman calculó el valor de cada casilla; la política es solo «ir hacia el
valor». Fíjate en que las flechas **se alejan de la trampa** incluso en casillas cercanas: el valor
negativo de la trampa se **propaga** a los vecinos. Eso es razonar sobre el futuro, no sobre el paso
inmediato.


## 5. Ver el valor y la política a la vez

Un mapa de calor lo hace evidente: las casillas calientes (claras) valen más; las frías (oscuras), menos.
Encima dibujamos la flecha de la política. Así se ve de un vistazo que el valor crece según te acercas a
la salida y se hunde cerca de la trampa.


In [ ]:
malla = np.full((FILAS, COLS), np.nan)
for (f,c), v in V.items(): malla[f,c] = v

fig, ax = plt.subplots(figsize=(5.2, 3.8))
im = ax.imshow(malla, cmap="Greys_r")
for f in range(FILAS):
    for c in range(COLS):
        s = (f,c)
        if s == MURO:
            ax.text(c, f, "#", ha="center", va="center", fontsize=16, color="red"); continue
        etq = {META:"+1", TRAMPA:"-1"}.get(s, POL.get(s, ""))
        ax.text(c, f, f"{etq}\n{V[s]:+.2f}", ha="center", va="center",
                color="white" if malla[f,c] < 0.4 else "black", fontsize=10)
ax.set_xticks([]); ax.set_yticks([]); ax.set_title("Valor (color) y politica (flecha)")
plt.colorbar(im, ax=ax, shrink=0.8, label="valor"); plt.tight_layout(); plt.show()


## 6. La trampa contamina a su vecina de abajo

Para ver la **propagación** del valor con números, miramos la fila de abajo entera. La casilla que está
justo **debajo** de la trampa vale menos que sus compañeras de fila, aunque ella misma no tenga castigo:
el agente «huele» el peligro de arriba. Y su política, en vez de subir (hacia la trampa), **huye** hacia
un lado. Eso es razonar sobre el futuro, no sobre el paso inmediato.


In [ ]:
print("Valor de la fila de abajo (columnas 0 a 3, de izquierda a derecha):")
print("   " + "   ".join(f"{V[(2,c)]:+.3f}" for c in range(COLS)))

print(f"\nLa casilla (2,3), justo DEBAJO de la trampa, vale {V[(2,3)]:+.3f}: es la mas baja de su fila.")
print(f"Su politica es '{POL[(2,3)]}': huye hacia un lado en vez de subir hacia la trampa.")
print(f"\nLa casilla (2,2), una mas a la izquierda y ya lejos del peligro, vale {V[(2,2)]:+.3f}.")
print("El castigo de la trampa se propaga hacia abajo y hunde el valor de su vecina.")


## 7. Experimento: el descuento γ cambia el carácter del agente

γ controla cuánto le importa al agente el futuro. Con γ alto (cerca de 1), piensa a largo plazo y acepta
rodeos para llegar bien. Con γ bajo, se vuelve cortoplacista: prefiere el premio inmediato y arriesga
más. Lo medimos resolviendo el mundo con varios γ y mirando cuánto vale la casilla de inicio y qué hace
en ella.


In [ ]:
def resuelve(gamma, coste=RECOMPENSA_PASO):
    V = {s: 0.0 for s in estados()}
    for s, r in terminales.items(): V[s] = r
    for _ in range(400):
        d = 0
        for s in estados():
            if s in terminales: continue
            mejor = max(valor_de(s, a, V, gamma, coste) for a in ACCIONES)
            d = max(d, abs(mejor - V[s])); V[s] = mejor
        if d < 1e-8: break
    return V, politica_de(V, gamma, coste)

print("gamma | valor del inicio (2,0) | politica del inicio")
for g in [0.99, 0.9, 0.5, 0.2, 0.1]:
    Vg, polg = resuelve(g)
    print(f"  {g:>4} |        {Vg[(2,0)]:+.3f}         |        {polg[(2,0)]}")
print("\nCon gamma alto el inicio vale mas (el futuro premio cuenta); con gamma bajo, el agente apenas")
print("valora llegar a la meta lejana: se vuelve cortoplacista.")


## 8. Experimento: el coste del paso moldea el comportamiento

El -0,04 de cada paso no es un detalle: es lo que le mete prisa al robot. Si **encarecemos** mucho cada
movimiento, el robot tendrá tanta urgencia por terminar que aceptará pasar más cerca de la trampa con tal
de acabar antes. Si lo hacemos casi gratis, se vuelve perezoso y prudente. Es el corazón del *reward
shaping*: las recompensas, mal puestas, producen comportamientos absurdos pero «óptimos».


In [ ]:
print("coste del paso | valor del inicio | casillas que cambian de decision respecto al mundo barato (-0.04)")
for coste in [-0.01, -0.04, -0.2, -0.5, -1.0]:
    Vc, polc = resuelve(GAMMA, coste)
    cambios = sum(1 for s in polc if polc[s] != POL[s])
    print(f"   {coste:>6}      |     {Vc[(2,0)]:+.3f}      |   {cambios} de {len(polc)} casillas")
print("\nCuanto mas caro el paso, mas baja el valor del inicio y mas casillas cambian su decision: el")
print("robot deja de dar rodeos prudentes porque cada paso de mas le sale carisimo.")


## 9. Experimento: ¿cuánto importa que el mundo resbale?

Hasta ahora el mundo es resbaladizo (80/10/10). ¿Y si fuera **determinista** (la acción siempre sale como
quieres) o **más caótico** todavía? La política prudente solo tiene sentido cuando hay azar: en un mundo
determinista, el robot puede pasar pegado a la trampa sin miedo, porque nunca resbala hacia ella. Medimos
cuántas casillas cambian su decisión respecto al mundo determinista a medida que metemos ruido.


In [ ]:
def resuelve_con_ruido(p_acierto):
    p_lado = (1 - p_acierto) / 2
    def res(s, a):
        perp = {"^":["<",">"], "v":["<",">"], "<":["^","v"], ">":["^","v"]}[a]
        return [(p_acierto, mover(s,a)), (p_lado, mover(s,perp[0])), (p_lado, mover(s,perp[1]))]
    V = {s: 0.0 for s in estados()}
    for s, r in terminales.items(): V[s] = r
    for _ in range(400):
        d = 0
        for s in estados():
            if s in terminales: continue
            mejor = max(sum(p*(RECOMPENSA_PASO + GAMMA*V[s2]) for p, s2 in res(s,a)) for a in ACCIONES)
            d = max(d, abs(mejor - V[s])); V[s] = mejor
        if d < 1e-8: break
    pol = {s: max(ACCIONES, key=lambda a: sum(p*(RECOMPENSA_PASO + GAMMA*V[s2]) for p, s2 in res(s,a)))
           for s in estados() if s not in terminales}
    return V, pol

Vdet, poldet = resuelve_con_ruido(1.0)
print("p(acierto) |  mundo            | valor del inicio | casillas distintas al mundo determinista")
for p in [1.0, 0.8, 0.6]:
    Vp, polp = resuelve_con_ruido(p)
    etiqueta = {1.0:"determinista", 0.8:"resbaladizo", 0.6:"muy caotico"}[p]
    dif = sum(1 for s in polp if polp[s] != poldet[s])
    print(f"    {p:>4}    | {etiqueta:<16} |     {Vp[(2,0)]:+.3f}      |   {dif} de {len(polp)} casillas")
print("\nCon menos azar el inicio vale mas y la politica se acerca a la del mundo determinista; con")
print("mucho azar aparecen cambios: el robot prefiere rutas con margen porque resbalar cerca de la")
print("trampa se paga muy caro.")


## 10. Simular trayectorias: la política, en acción

Tener la política es una cosa; *verla funcionar*, otra. Soltamos al robot muchas veces desde el inicio,
dejando que el mundo resbaladizo haga de las suyas, y contamos cuántas veces llega a la salida, cuántas
cae en la trampa y cuántos pasos tarda de media. Como hay azar, ningún viaje es igual: la política no
garantiza llegar, **maximiza la probabilidad** de llegar bien.


In [ ]:
np.random.seed(0)
def un_viaje(pol, inicio=(2,0), max_pasos=100):
    s = inicio
    for paso in range(1, max_pasos+1):
        if s in terminales: return ("meta" if s == META else "trampa"), paso
        probs, destinos = zip(*resultados(s, pol[s]))
        s = destinos[np.random.choice(len(destinos), p=probs)]
    return "perdido", max_pasos

N = 5000
finales = [un_viaje(POL) for _ in range(N)]
metas = sum(1 for r,_ in finales if r == "meta")
trampas = sum(1 for r,_ in finales if r == "trampa")
pasos_meta = [p for r,p in finales if r == "meta"]
print(f"De {N} viajes con la politica optima:")
print(f"  llega a la SALIDA: {metas} ({100*metas/N:.1f}%)")
print(f"  cae en la TRAMPA:  {trampas} ({100*trampas/N:.1f}%)")
print(f"  pasos medios hasta la salida: {np.mean(pasos_meta):.1f}")
print("\nNi siquiera la politica optima evita la trampa el 100% de las veces: el azar manda.")
print("Pero la inmensa mayoria de los viajes acaban bien, que es lo que Bellman maximiza.")


## 11. Policy iteration: el mismo destino por otro camino

*Value iteration* mezcla en cada vuelta «evaluar» y «mejorar». **Policy iteration** las separa: parte de
una política cualquiera, calcula **exactamente** cuánto vale seguirla (evaluación), y luego la **mejora**
casilla a casilla; repite hasta que deja de cambiar. Suele converger en muy pocas vueltas. Lo bonito es
que llega a la **misma** política óptima: dos algoritmos, un solo punto fijo.


In [ ]:
def policy_iteration(gamma=GAMMA, coste=RECOMPENSA_PASO):
    pol = {s: ">" for s in estados() if s not in terminales}   # politica inicial tonta
    for vuelta in range(1, 100):
        # 1) evaluar la politica actual (iterando hasta estabilizar V)
        V = {s: 0.0 for s in estados()}
        for s, r in terminales.items(): V[s] = r
        for _ in range(1000):
            d = 0
            for s in pol:
                v = valor_de(s, pol[s], V, gamma, coste)
                d = max(d, abs(v - V[s])); V[s] = v
            if d < 1e-9: break
        # 2) mejorar la politica siendo avariciosos respecto a V
        estable = True
        for s in pol:
            mejor_a = max(ACCIONES, key=lambda a: valor_de(s, a, V, gamma, coste))
            if mejor_a != pol[s]: estable = False
            pol[s] = mejor_a
        if estable: return pol, V, vuelta
    return pol, V, vuelta

pol_pi, V_pi, vueltas = policy_iteration()
iguales = all(pol_pi[s] == POL[s] for s in pol_pi)
print(f"Policy iteration convergio en {vueltas} mejoras de politica.")
print(f"Value iteration habia necesitado {iteracion} barridos de Bellman.")
print(f"\nLas dos politicas coinciden en todas las casillas: {iguales}")
print("\nDos algoritmos distintos, el mismo punto fijo. Asi es Bellman: el optimo es unico.")


## 12. Pruébalo tú

1. **Sube el coste del paso** a `-0,2` en la celda inicial y reejecuta: el robot tendrá tanta prisa que
   aceptará más riesgo. Compáralo con la tabla del experimento 8.
2. **Haz el mundo determinista** (`resuelve_con_ruido(1.0)`): ¿se atreve a pasar pegado a la trampa? La
   incertidumbre cambia lo que es prudente.
3. **Mueve la trampa** junto a la meta (cambia `TRAMPA`) y observa en el mapa de calor cómo la política
   rodea con más cuidado.
4. **Agranda el mundo** a 4×5 con varias trampas y mira si *value iteration* sigue convergiendo igual de
   rápido en la curva del paso 3.
5. **Sube γ a 1,0** y observa: sin descuento, en un mundo con casillas terminales aún converge, pero el
   valor del inicio se dispara. Prueba a quitar las terminales y verás que deja de converger.


## 13. Errores comunes

- **Olvidar el azar de las transiciones.** En el mundo real, las acciones no siempre salen como quieres;
  la política óptima tiene eso en cuenta. La simulación del paso 10 lo deja claro: ni el óptimo gana
  siempre.
- **Poner γ = 1 sin pensar.** Sin descuento, en problemas sin fin los valores pueden no converger. γ < 1
  da convergencia y modela «el futuro vale menos».
- **Diseñar mal las recompensas.** Es el verdadero arte (*reward engineering*, capítulo 6): el
  experimento 8 muestra cómo un coste mal puesto cambia por completo el comportamiento.
- **Confundir valor y recompensa.** La recompensa es inmediata; el valor mira todo el futuro esperado. La
  trampa contamina a sus vecinos (paso 6) precisamente porque el valor mira hacia delante.
- **Creer que value iteration y policy iteration dan cosas distintas.** Dan la misma política óptima;
  cambian el camino y el coste de cálculo, no el destino.


## 14. Qué te llevas

- Un **MDP** es estados, acciones, recompensas, un poco de azar y un descuento; resolverlo es encontrar la
  **política** que maximiza el retorno a largo plazo.
- La **ecuación de Bellman** liga el valor de un estado con el de los que le siguen; iterarla hace emerger
  la política sin programar el camino, y converge a ritmo geométrico (lo viste en la curva).
- Las **recompensas** son el lenguaje con el que defines lo que quieres, el **descuento** γ fija el
  horizonte y el **azar** del entorno decide cuánta prudencia conviene: diseñarlos bien es el verdadero
  arte (capítulo 6).
- **Value iteration** y **policy iteration** son dos rutas al mismo óptimo; simular trayectorias confirma
  que la política maximiza la probabilidad de acabar bien, no la certeza.

**Para seguir:** el siguiente cuaderno ataca el otro gran problema del refuerzo: cuando ni siquiera sabes
las recompensas y tienes que **explorar** para descubrirlas (bandits).


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*